In [ ]:
import os
import torch
import json
import matplotlib.pyplot as plt
import random
from spatialca import map, game
import math

## Data

In [ ]:
COLOUR_MAP = map.satellite_colour_map
num_classes = len(COLOUR_MAP)
_scale_down = 3
_n_splits = 3
num_agents = 20

_place = "rio"
og_img_dir = f"./openearth/{_place}/images"
img_dir = f"./openearth/{_place}/labels"
data_file_path = f"./data/openearth_map_train_data_{_place}.json"

real_images = [f"{og_img_dir}/{f}" for f in os.listdir(og_img_dir)]
seg_images = [f"{img_dir}/{f}" for f in os.listdir(img_dir)]

real_image_paths = real_images[0:2]
train_image_paths = seg_images[0:2]
test_image_paths = seg_images[2:3]

In [ ]:
real_images = [map.load_rgb_image(image_path) for image_path in real_image_paths]
fig, axes = plt.subplots(1, len(real_images), figsize=(len(real_images) * 4, 4))

for i, rgb_image in enumerate(real_images):
    axes[i].imshow(rgb_image, cmap="gray")
    axes[i].set_title(f"Real {_place.capitalize()} {i + 1}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

seg_images = [
    map.load_openearth_seg_image(image_path, color_map=COLOUR_MAP)
    for image_path in train_image_paths
]
fig, axes = plt.subplots(1, len(seg_images), figsize=(len(seg_images) * 4, 4))

for i, rgb_image in enumerate(seg_images):
    axes[i].imshow(rgb_image, cmap="gray")
    axes[i].set_title(f"Seg. {_place.capitalize()} {i + 1}")

plt.tight_layout()
plt.show()

train_images = [
    map.load_openearth_seg_image(
        image_path, scale_down=_scale_down, color_map=COLOUR_MAP
    )
    for image_path in train_image_paths
]
train_images = [
    img for image in train_images for img in map.split_image(image, _n_splits)
]
rows = (len(train_images) + 7) // 9  # Calculate the number of rows needed
fig, axes = plt.subplots(rows, 9, figsize=(32, rows * 4))

for idx, rgb_image in enumerate(train_images):
    row, col = divmod(idx, 9)
    axes[row, col].imshow(rgb_image, cmap="gray")
    axes[row, col].set_title(f"Train Seg. {_place.capitalize()} {idx + 1}")

plt.tight_layout()
plt.show()

In [ ]:
COLOUR_MAP = map.synthetic_colours
num_classes = len(COLOUR_MAP)
cities = ["archipelago", "river", "farmland"]
n_cities = len(cities)
num_agents = 10
data_file_path = "./data/map_train_data_map_train_data_synth_all.json"

In [ ]:
real_images = [
    map.generate_synthetic_satellite_image(city, COLOUR_MAP) for city in cities
]
fig, axes = plt.subplots(1, n_cities, figsize=(n_cities * 3, 3))
n_cols = 7

for i, (rgb_image, city) in enumerate(zip(real_images, cities)):
    axes[i].imshow(rgb_image, cmap="gray")
    axes[i].set_title(city)
    axes[i].axis("off")

plt.tight_layout()
plt.show()

cities = (cities * math.ceil(14 / 3))[:14]
train_images = [
    map.generate_synthetic_satellite_image(city, COLOUR_MAP) for city in cities
]

rows = (len(train_images) + n_cols - 1) // n_cols  # Calculate the number of rows needed
fig, axes = plt.subplots(rows, n_cols, figsize=(32, rows * 4))

for idx, (rgb_image, city) in enumerate(zip(train_images, cities)):
    row, col = divmod(idx, n_cols)
    axes[row, col].imshow(rgb_image, cmap="gray")
    axes[row, col].set_title(f"{city}, image {idx + 1}")

plt.tight_layout()
plt.show()

In [ ]:
n_rollouts, n_steps = 2, 150
if not os.path.exists(data_file_path):
    train_data = []
    for img_index, train_img in enumerate(train_images):
        print(f"Starting image {img_index + 1}/{len(train_images)}")
        train_img = torch.from_numpy(train_img)
        map_snapshots = game.MultiAgentMapper(
            raw_map=train_img,
            num_agents=num_agents,
            movement_style="frontier",
            colour_map=COLOUR_MAP,
            agent_view_radius=2,
        ).collect_snapshot_data(rollouts=n_rollouts, steps=n_steps, interval=20)

        map_snapshots_np = [
            (sample[0].tolist(), train_img.tolist())
            for sample in map_snapshots
            if 0.1 < sample[0][:, :, 3].mean() < 0.9
        ]
        train_data.extend(map_snapshots_np)

    with open(data_file_path, "w") as f:
        json.dump(train_data, f)

In [ ]:
with open(data_file_path, "r") as f:
    train_data = [
        (torch.tensor(sample[0]), torch.tensor(sample[1])) for sample in json.load(f)
    ]

In [ ]:
# showing representative samples
bs_to_show = 8
batch = random.sample(train_data, bs_to_show)
fig, axes = plt.subplots(2, bs_to_show, figsize=(bs_to_show * 2, bs_to_show))
for i, (sample, target) in enumerate(batch):
    axes[0, i].imshow(sample[:, :, :3])
    axes[0, i].set_title(f"Sample {i + 1}")
    axes[0, i].axis("off")
    axes[1, i].imshow(target)
    axes[1, i].set_title(f"Target {i + 1}")
    axes[1, i].axis("off")
plt.tight_layout()

In [ ]:
all_samples = torch.stack([sample for sample, _ in train_data])
percent_missing = [
    (sample[:, :, 3] == 0).float().mean().item() for sample in all_samples
]
plt.figure(figsize=(12, 4))
plt.hist(percent_missing, bins=5, color="skyblue", edgecolor="black")
plt.grid(axis="y", alpha=0.75)
plt.xlabel("Missing % per Sample")
plt.show()

In [ ]:
all_training_data = []
for sample, target in train_data:
    input_oh = map.rgb_to_input_tensor(sample, COLOUR_MAP)
    target_oh = map.rgb_to_target_tensor(target, COLOUR_MAP)
    all_training_data.append(
        (
            input_oh[:num_classes],
            input_oh[num_classes : num_classes + 1],
            target_oh,
            target_oh.argmax(dim=0),
        )
    )

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(6, 6))
index = 0

(
    sample,
    target,
) = train_data[index]
input_oh, mask, target_oh, target_class_idx = all_training_data[index]

axes[0, 0].imshow(sample[:, :, :3])  # Display the RGB part of the sample
axes[0, 0].set_title("Sample")
axes[0, 0].axis("off")

axes[0, 1].imshow(sample[:, :, 3], cmap="gray")  # Display the mask
axes[0, 1].set_title("Mask")
axes[0, 1].axis("off")

axes[1, 0].imshow(target[:, :, :3])  # Display the RGB part of the target
axes[1, 0].set_title("Target")
axes[1, 0].axis("off")

axes[1, 1].imshow(target_oh.argmax(dim=0), cmap="viridis")
axes[1, 1].set_title("Target Argmax")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

n_channels = input_oh.shape[0]
fig, axes = plt.subplots(1, n_channels, figsize=(12, 3))
for c in range(n_channels):
    _name = f"Class {c + 1}"
    axes[c].imshow(input_oh[c], cmap="gray")
    axes[c].set_title(_name)
    axes[c].axis("off")
plt.tight_layout()
plt.show()